<a href="https://colab.research.google.com/github/Shineii86/LeechBot/blob/main/notebooks/LeechBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
<img src="https://capsule-render.vercel.app/api?type=waving&height=300&color=gradient&text=𝗟𝗲𝗲𝗰𝗵%20𝗕𝗼𝘁&fontAlignY=30&fontSize=100&desc=𝖠𝖽𝗏𝖺𝗇𝖼𝖾𝖽%20𝖳𝖾𝗅𝖾𝗀𝗋𝖺𝗆%20𝖥𝗂𝗅𝖾%20𝖳𝗋𝖺𝗇𝗌𝗅𝗈𝖺𝖽𝖾𝗋&descSize=30" />

**A powerful Pyrogram-based bot to transfer files to Telegram & Google Drive**

![Version](https://img.shields.io/badge/Version-3.1.5-8B5CF6?style=for-the-badge)
![Python](https://img.shields.io/badge/Python-3.10+-3776AB?style=for-the-badge&logo=python&logoColor=white)
![License](https://img.shields.io/badge/License-MIT-06B6D4?style=for-the-badge)

---

### ✨ Features

| 📥 Download From | 📤 Upload To | 🛠️ Tools |
|:---:|:---:|:---:|
| YouTube, Facebook, Instagram | Telegram | Video Converter (GPU) |
| Google Drive, Mega, Terabox | Google Drive | Archive Handler |
| Pixeldrain, Mediafire, Direct | Directory Leech | Smart Splitting |
| 2000+ sites via yt-dlp | Batch Photos | Download Queue |

---

### 🚀 Quick Start

1. **Fill credentials** in Cell 2 (or use Colab Secrets)
2. Click **Runtime → Run all** or press **Ctrl+F9**
3. Bot starts automatically — send `/start` on Telegram

---

### 📋 Cells

| # | Cell | Purpose |
|:--|:-----|:--------|
| 1 | ♻️ Google Drive Setup | Optional: mount Google Drive |
| 2 | 📦 Setup LeechBot | Clone repo, install deps, configure |
| 3 | 🚀 Deploy LeechBot | Start bot + dashboard tunnel + keep-alive |
| 4 | 🔄 Update LeechBot | Pull latest changes |
| 5 | 🔍 Health Check | Pre-flight diagnostics |

</div>

In [ ]:
# @title ♻️ Google Drive Setup
#@markdown <div align="center">
#@markdown <img src="https://user-images.githubusercontent.com/125879861/255377947-6ac19c35-dbbd-4a9b-bc0e-c603de81c533.png" height="60">
#@markdown <h4>Google Drive Integration</h4>
#@markdown </div>

ACTION = "Mount Drive" # @param ["Mount Drive", "Unmount Drive", "Generate Token", "Skip"]
MOUNT_PATH = "/content/drive" # @param {type:"string"}
TOKEN_PATH = "/content/token.pickle" # @param {type:"string"}

import os, time, pickle
from IPython.display import display, Markdown, clear_output

def log(emoji, msg, color="cyan"):
    display(Markdown(f"<font color={color}>**{emoji} {msg}**</font>"))

if ACTION == "Skip":
    log("⏭️", "Skipped — Google Drive not needed", "gray")
else:
    from google.colab import auth, drive
    import google.auth
    from google.auth.transport.requests import Request

    if ACTION == "Mount Drive":
        for attempt in range(3):
            try:
                log("🔗", f"Mounting to {MOUNT_PATH}... (attempt {attempt+1}/3)")
                drive.mount(MOUNT_PATH, force_remount=True)
                log("✅", "Drive mounted!", "green")
                break
            except Exception as e:
                log("⚠️", f"Attempt {attempt+1} failed: {e}", "orange")
                time.sleep(2)

    elif ACTION == "Unmount Drive":
        try:
            drive.flush_and_unmount()
            log("🔓", "Drive unmounted", "green")
        except Exception as e:
            log("⚠️", f"Unmount: {e}", "orange")

    elif ACTION == "Generate Token":
        try:
            auth.authenticate_user()
            creds, _ = google.auth.default()
            if creds.expired and creds.refresh_token:
                creds.refresh(Request())
            with open(TOKEN_PATH, 'wb') as f:
                pickle.dump(creds, f)
            os.chmod(TOKEN_PATH, 0o600)
            log("✅", f"Token saved to {TOKEN_PATH}", "green")
        except Exception as e:
            log("❌", f"Token error: {e}", "red")

    if ACTION == "Mount Drive":
        try:
            auth.authenticate_user()
            creds, _ = google.auth.default()
            if creds.expired and creds.refresh_token:
                creds.refresh(Request())
            with open(TOKEN_PATH, 'wb') as f:
                pickle.dump(creds, f)
            os.chmod(TOKEN_PATH, 0o600)
            log("🔑", "GDrive token generated", "green")
        except Exception as e:
            log("⚠️", f"Token gen skipped: {e}", "orange")

log("💡", "Tip: Store credentials in Colab Secrets for auto-fill", "gray")

In [ ]:
# @title 📦 Setup LeechBot
#@markdown <div align="center">
#@markdown <img src="https://user-images.githubusercontent.com/125879861/255391401-371f3a64-732d-4954-ac0f-4f093a6605e1.png" width="500">
#@markdown </div>

#@markdown ---
#@markdown ## 🔐 Credentials
#@markdown > Fill manually or set in **🔑 Secrets** (left panel) with these names:
#@markdown >
#@markdown > `LEECHBOT_API_ID` · `LEECHBOT_API_HASH` · `LEECHBOT_BOT_TOKEN` · `LEECHBOT_USER_ID` · `LEECHBOT_DUMP_ID`

API_ID = 0 # @param {type:"integer"}
API_HASH = "" # @param {type:"string"}
BOT_TOKEN = "" # @param {type:"string"}
OWNER_ID = 0 # @param {type:"integer"}
DUMP_ID = 0 # @param {type:"integer"}

#@markdown ---
#@markdown ## ⚙️ Options
MOUNT_DRIVE = False # @param {type:"boolean"}
USE_GPU = True # @param {type:"boolean"}
REPO_BRANCH = "main" # @param ["main"]

# ═══════════════════════════════════════════════════════════
# 📦 Setup Engine
# ═══════════════════════════════════════════════════════════

import subprocess, sys, os, json, time, shutil
from pathlib import Path
from IPython.display import clear_output, display, Markdown
import logging

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["ALSA_CONFIG_PATH"] = "/dev/null"

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger("LeechBot")

# ─── UI Helpers ───────────────────────────────────────────
STEP = [0]

def banner():
    return """
    ╔═══════════════════════════════════════════╗
    ║         🚀 L E E C H B O T               ║
    ║    Advanced Telegram File Transloader     ║
    ╠═══════════════════════════════════════════╣
    ║  👤 Shinei Nouzen  ·  📂 Shineii86       ║
    ╚═══════════════════════════════════════════╝
    """

def log(emoji, msg, color="#2196F3"):
    display(Markdown(f'<font color="{color}">**{emoji} {msg}**</font>'))

def step(msg):
    STEP[0] += 1
    display(Markdown(f"\n---\n### Step {STEP[0]}: {msg}"))

def ok(msg):   log("✅", msg, "#4CAF50")
def fail(msg): log("❌", msg, "#F44336")
def warn(msg): log("⚠️", msg, "#FF9800")
def info(msg): log("ℹ️", msg, "#2196F3")

def run(cmd, desc, retries=3):
    for i in range(retries):
        try:
            info(f"{desc} (attempt {i+1}/{retries})")
            r = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True, timeout=300)
            ok(f"{desc} — done")
            return True
        except subprocess.CalledProcessError as e:
            if i == retries - 1:
                fail(f"{desc} failed: {e.stderr[:200]}")
                return False
            time.sleep(2 ** i)
        except subprocess.TimeoutExpired:
            if i == retries - 1:
                fail(f"{desc} timed out")
                return False
    return False

# ─── Credentials ──────────────────────────────────────────
def load_credentials():
    creds = {}
    try:
        from google.colab import userdata
        secrets = {
            'API_ID': 'LEECHBOT_API_ID', 'API_HASH': 'LEECHBOT_API_HASH',
            'BOT_TOKEN': 'LEECHBOT_BOT_TOKEN', 'OWNER_ID': 'LEECHBOT_USER_ID',
            'DUMP_ID': 'LEECHBOT_DUMP_ID'
        }
        for key, name in secrets.items():
            try:
                val = userdata.get(name)
                creds[key] = int(val) if key in ['API_ID', 'OWNER_ID', 'DUMP_ID'] else val
                ok(f"{key} loaded from Colab Secrets")
            except:
                creds[key] = None
    except ImportError:
        pass

    fallbacks = {'API_ID': API_ID, 'API_HASH': API_HASH, 'BOT_TOKEN': BOT_TOKEN,
                 'OWNER_ID': OWNER_ID, 'DUMP_ID': DUMP_ID}
    for k in fallbacks:
        if not creds.get(k):
            creds[k] = fallbacks[k]
    return creds

def validate(creds):
    required = ['API_ID', 'API_HASH', 'BOT_TOKEN', 'OWNER_ID', 'DUMP_ID']
    missing = [k for k in required if not creds.get(k)]
    if missing:
        fail(f"Missing: {', '.join(missing)}")
        return False
    d = str(creds['DUMP_ID'])
    if len(d) == 10 and not d.startswith('-100'):
        creds['DUMP_ID'] = int(f"-100{d}")
        info("Auto-formatted DUMP_ID with -100 prefix")
    return True

# ─── Setup ───────────────────────────────────────────────
def setup():
    clear_output(wait=True)
    print(banner())

    # Step 1: Credentials
    step("🔐 Load Credentials")
    creds = load_credentials()
    if not validate(creds):
        fail("Fix credentials and re-run this cell")
        return

    # Step 2: Clone
    step("📦 Clone Repository")
    os.chdir("/content")
    if os.path.exists("/content/leechbot"):
    # Persist credentials across re-clones
    _cred_backup = "/content/.leechbot_creds.json"
    _existing_creds = os.path.join("/content/leechbot", "credentials.json")
    if os.path.exists(_existing_creds):
        import shutil as _shutil
        _shutil.copy2(_existing_creds, _cred_backup)
        info("Backed up existing credentials")

        shutil.rmtree("/content/leechbot")
        info("Cleaned previous install")
    if not run(f"git clone -b {REPO_BRANCH} --depth 1 https://github.com/Shineii86/LeechBot.git /content/leechbot",
               "Cloning LeechBot"):
        return

    # Step 3: Dependencies
    step("📦 Install Dependencies")
    if not run("apt-get update -qq && apt-get install -y -qq ffmpeg aria2 megatools p7zip-full unzip",
               "System packages"):
        return
    if not run("pip3 install -q --no-cache-dir -r /content/leechbot/requirements.txt",
               "Python packages"):
        return

    # Step 4: GPU Check
    step("🎮 Hardware Check")
    if USE_GPU:
        try:
            r = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                              shell=True, capture_output=True, text=True, check=True)
            name, mem = r.stdout.strip().split(', ')
            ok(f"GPU: {name} ({mem} VRAM)")
        except:
            info("No GPU — using CPU")
    else:
        info("GPU disabled by user")

    # Step 5: Save config
    step("💾 Save Configuration")
    cfg_path = "/content/leechbot/credentials.json"
    with open(cfg_path, 'w') as f:
        json.dump(creds, f, indent=2)
    os.chmod(cfg_path, 0o600)
    ok("credentials.json saved")

    os.environ["API_ID"] = str(creds["API_ID"])
    os.environ["API_HASH"] = str(creds["API_HASH"])
    os.environ["BOT_TOKEN"] = str(creds["BOT_TOKEN"])
    os.environ["OWNER_ID"] = str(creds["OWNER_ID"])
    os.environ["DUMP_ID"] = str(creds["DUMP_ID"])
    ok("Environment variables set")

    # Step 6: Mount Drive (optional)
    if MOUNT_DRIVE:
        step("☁️ Mount Google Drive")
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            ok("Drive mounted")
        except Exception as e:
            warn(f"Drive mount failed: {e}")

    # Step 7: Clean sessions
    step("🧹 Clean Sessions")
    for sf in ["/content/leechbot/leechbot_session.session",
               "/content/leechbot/leechbot_session.session-journal"]:
        if os.path.exists(sf):
            os.remove(sf)
            info(f"Removed: {sf}")
    ok("Sessions cleaned")

    ok("Setup complete! Run the **🚀 Launch Bot** cell next.")

# ─── Run ──────────────────────────────────────────────────
try:
    setup()
except KeyboardInterrupt:
    warn("Cancelled by user")
except Exception as e:
    fail(f"Unexpected error: {e}")
    logger.exception("Full traceback:")


In [ ]:
# @title 🚀 Deploy LeechBot
#@markdown Start the bot, set up dashboard tunnel, and keep the session alive.
#@markdown > Run the **📦 Setup** cell first!

#@markdown ---
#@markdown ## 🌐 Dashboard Tunnel
TUNNEL_METHOD = "ngrok" # @param ["ngrok", "cloudflared", "Skip"]
#@markdown > **ngrok** — reliable, needs free token · **cloudflared** — no signup, random URL
NGROK_TOKEN = "" # @param {type:"string"}
#@markdown > Get your token from https://dashboard.ngrok.com/get-started/your-authtoken

#@markdown ---
#@markdown ## ⚙️ Options
AUTO_RESTART = True # @param {type:"boolean"}

# ═══════════════════════════════════════════════════════════
# 🚀 Deploy + Tunnel + Keep-Alive Engine
# ═══════════════════════════════════════════════════════════

import subprocess, sys, os, time, threading, datetime, signal
from IPython.display import clear_output, display, Markdown, Javascript

BOT_LOG = "/content/leechbot/bot.log"
BOT_DIR = "/content/leechbot"
WEB_PORT = os.environ.get("WEB_PORT", "8080")
bot_proc = None

# ─── UI Helpers ───────────────────────────────────────────
def banner():
    return """
    ╔═══════════════════════════════════════════╗
    ║         🚀 L E E C H B O T               ║
    ║    Advanced Telegram File Transloader     ║
    ╠═══════════════════════════════════════════╣
    ║  👤 Shinei Nouzen  ·  📂 Shineii86       ║
    ╚═══════════════════════════════════════════╝
    """

def log(emoji, msg, color="#2196F3"):
    display(Markdown(f'<font color="{color}">**{emoji} {msg}**</font>'))

def ok(msg):   log("✅", msg, "#4CAF50")
def fail(msg): log("❌", msg, "#F44336")
def warn(msg): log("⚠️", msg, "#FF9800")
def info(msg): log("ℹ️", msg, "#2196F3")

# ─── Step 1: Launch Bot ───────────────────────────────────
clear_output(wait=True)
print(banner())

if not os.path.exists(f"{BOT_DIR}/leechbot/__init__.py"):
    fail("LeechBot not installed. Run the 📦 Setup cell first.")
else:
    os.chdir(BOT_DIR)

    # Kill stale processes on port
    subprocess.run(["fuser", "-k", f"{WEB_PORT}/tcp"], capture_output=True)
    time.sleep(1)

    # Start bot
    info("Starting LeechBot...")
    log_fh = open(BOT_LOG, "w")
    bot_proc = subprocess.Popen(
        [sys.executable, "-m", "leechbot"],
        stdout=log_fh,
        stderr=subprocess.STDOUT,
        cwd=BOT_DIR
    )

    # Wait for startup
    startup_ok = False
    for _ in range(60):
        time.sleep(1)
        if bot_proc.poll() is not None:
            fail("Bot crashed on startup!")
            with open(BOT_LOG) as f:
                print(f.read()[-3000:])
            bot_proc = None
            break
        try:
            with open(BOT_LOG) as f:
                if "LeechBot started successfully" in f.read():
                    startup_ok = True
                    break
        except:
            pass

    if not startup_ok and bot_proc is not None:
        fail("Bot did not start within 60 seconds")
        with open(BOT_LOG) as f:
            print(f.read()[-3000:])
        bot_proc = None

    # ─── Step 2: Dashboard Tunnel ─────────────────────────
    if bot_proc is not None:
        clear_output(wait=True)
        print(banner())
        ok("LeechBot is running!")

        if TUNNEL_METHOD != "Skip":
            info(f"Setting up {TUNNEL_METHOD} tunnel...")

            if TUNNEL_METHOD == "ngrok":
                try:
                    from pyngrok import ngrok
                except ImportError:
                    info("Installing pyngrok...")
                    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
                    from pyngrok import ngrok

                if not NGROK_TOKEN:
                    try:
                        from google.colab import userdata
                        NGROK_TOKEN = userdata.get('NGROK_TOKEN')
                        ok("Ngrok token loaded from Colab Secrets")
                    except:
                        fail("No ngrok token! Get one at https://dashboard.ngrok.com/get-started/your-authtoken")
                        NGROK_TOKEN = None

                if NGROK_TOKEN:
                    ngrok.set_auth_token(NGROK_TOKEN)
                    ngrok.kill()
                    time.sleep(1)
                    tunnel = ngrok.connect(int(WEB_PORT), bind_tls=True)
                    public_url = tunnel.public_url
                    display(Markdown(f"""
#### ✅ Dashboard Ready (ngrok)

| | |
|:---|:---|
| **🌐 URL** | [{public_url}]({public_url}) |
| **🔑 Token** | Check `🔑 Auth token:` in bot logs below |
| **📡 Status** | Live |

> Open the URL above → enter the token → Connect
"""))

            elif TUNNEL_METHOD == "cloudflared":
                import shutil as sh
                if not sh.which("cloudflared"):
                    info("Installing cloudflared...")
                    subprocess.run(
                        "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
                        shell=True, check=True
                    )
                    ok("cloudflared installed")

                display(Markdown("""
> Look for the `.trycloudflare.com` URL in the output below.
> Copy it → open in browser → enter the auth token.
"""))
                def _tunnel():
                    os.system(f"cloudflared tunnel --url http://localhost:{WEB_PORT}")
                threading.Thread(target=_tunnel, daemon=True).start()

        # Show bot commands
        display(Markdown("""
| Command | Action |
|:--------|:-------|
| `/start` | Initialize bot |
| `/tupload` | Leech to Telegram |
| `/gdupload` | Mirror to Google Drive |
| `/ytupload` | YouTube / yt-dlp download |
| `/settings` | Bot preferences |
| `/help` | All commands |
"""))

    # ─── Step 3: Keep-Alive + Auto-Restart ─────────────────
    if bot_proc is not None:
        # Inject multi-strategy JS keep-alive
        display(Javascript('''
        function keepAlive() {
            // Click runtime indicators
            var btn = document.querySelector('#connect');
            if (btn) btn.click();
            var nb = document.querySelector('colab-notebook');
            if (nb && nb.shadowRoot) {
                var ind = nb.shadowRoot.querySelector('#runtime-indicator');
                if (ind) ind.click();
            }
            // Simulate DOM activity
            window.scrollBy(0, 1); window.scrollBy(0, -1);
            document.dispatchEvent(new MouseEvent('mousemove', {
                clientX: Math.random() * window.innerWidth,
                clientY: Math.random() * window.innerHeight
            }));
            document.dispatchEvent(new KeyboardEvent('keydown', {key: 'Shift', keyCode: 16}));
            // Focus/blur cycle
            window.dispatchEvent(new Event('focus'));
            setTimeout(function(){ window.dispatchEvent(new Event('blur')); }, 100);
            setTimeout(function(){ window.dispatchEvent(new Event('focus')); }, 200);
        }
        setInterval(keepAlive, 30000);
        document.addEventListener('visibilitychange', function() {
            if (!document.hidden) keepAlive();
        });
        console.log('🛡️ LeechBot keep-alive active (30s interval)');
        '''))

        info("Keep-alive active — this cell stays alive to prevent disconnect")
        info("Bot is running. Ctrl+C to stop.")

        # Monitor + auto-restart loop
        restart_count = 0
        MAX_RESTARTS = 5
        start_time = time.time()

        def get_bot_pid():
            try:
                r = subprocess.run(["pgrep", "-f", "python3 -m leechbot"],
                                  capture_output=True, text=True)
                if r.returncode == 0:
                    return int(r.stdout.strip().split()[0])
            except: pass
            return None

        def restart_bot():
            global restart_count, bot_proc
            restart_count += 1
            print(f"\n🔄 Restarting bot ({restart_count}/{MAX_RESTARTS})...")
            subprocess.run(["fuser", "-k", f"{WEB_PORT}/tcp"], capture_output=True)
            time.sleep(2)
            fh = open(BOT_LOG, "a")
            fh.write(f"\n\n{'='*40}\nAuto-restart #{restart_count} at {datetime.datetime.now()}\n{'='*40}\n\n")
            fh.close()
            fh = open(BOT_LOG, "a")
            bot_proc = subprocess.Popen(
                [sys.executable, "-m", "leechbot"],
                stdout=fh, stderr=subprocess.STDOUT, cwd=BOT_DIR
            )
            time.sleep(10)
            return bot_proc.poll() is None

        try:
            while True:
                time.sleep(15)
                elapsed = int(time.time() - start_time)
                h, m, s = elapsed // 3600, (elapsed % 3600) // 60, elapsed % 60
                pid = get_bot_pid()
                if pid:
                    last_log = ""
                    try:
                        with open(BOT_LOG) as f:
                            lines = [l.strip() for l in f.readlines() if l.strip()]
                            last_log = lines[-1][:60] if lines else ""
                    except: pass
                    ts = datetime.datetime.now().strftime("%H:%M:%S")
                    print(f"\r💓 [{ts}] up {h:02d}:{m:02d}:{s:02d} | pid={pid} | {last_log:<60s}",
                          end="", flush=True)
                else:
                    print(f"\n⚠️ Bot process dead! (up {h:02d}:{m:02d}:{s:02d})")
                    if AUTO_RESTART and restart_count < MAX_RESTARTS:
                        if restart_bot():
                            ok("Bot restarted")
                        else:
                            fail("Restart failed")
                    elif restart_count >= MAX_RESTARTS:
                        fail(f"Max restarts ({MAX_RESTARTS}) reached.")
                        break
                    else:
                        fail("Auto-restart disabled.")
                        break
        except KeyboardInterrupt:
            print("\n")
            info("Stopping bot...")
            if bot_proc:
                bot_proc.terminate()
                try: bot_proc.wait(timeout=10)
                except: bot_proc.kill()
            ok("Bot stopped")


In [ ]:
# @title 🔄 Update LeechBot
#@markdown Pull the latest changes without re-cloning.
#@markdown > Use this when you see a new version on GitHub.

import subprocess, os
from IPython.display import display, Markdown

def log(emoji, msg, color="#2196F3"):
    display(Markdown(f'<font color="{color}">**{emoji} {msg}**</font>'))

REPO = "/content/leechbot"

if not os.path.exists(f"{REPO}/.git"):
    log("❌", "LeechBot not found. Run the Deployer cell first.", "red")
else:
    os.chdir(REPO)

    # Pull latest
    log("📥", "Pulling latest changes...")
    r = subprocess.run("git pull origin main", shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        output = r.stdout.strip()
        if "Already up to date" in output:
            log("✅", "Already up to date!", "green")
        else:
            log("✅", f"Updated:\n```\n{output}\n```", "green")

            # Re-install deps
            log("📦", "Checking dependencies...")
            subprocess.run("pip3 install -q --no-cache-dir -r requirements.txt",
                          shell=True, capture_output=True)
            log("✅", "Dependencies updated", "green")

            # Restart hint
            display(Markdown("""
---
### ⚠️ Restart Required

The code was updated. You need to restart the bot:

1. **Stop the running bot** (click ⏹️ in the running cell)
2. **Re-run the Deployer cell** — or run:

```python
!cd /content/leechbot && python3 -m leechbot
```
"""))
    else:
        log("❌", f"Git pull failed: {r.stderr[:200]}", "red")
        log("💡", "Try re-running the Deployer cell for a fresh install", "orange")

In [ ]:
# @title 🔍 Health Check
#@markdown Check if everything is set up correctly before deploying.

import os, subprocess, json
from IPython.display import display, Markdown

def check(label, ok, detail=""):
    icon = "✅" if ok else "❌"
    suffix = f" — `{detail}`" if detail else ""
    display(Markdown(f"{icon} **{label}**{suffix}"))

display(Markdown("## 🔍 Pre-Flight Check\n"))

# Python
v = sys.version.split()[0]
check("Python 3.10+", tuple(int(x) for x in v.split('.')) >= (3, 10), v)

# Node.js (needed for PO token plugin)
try:
    nj = subprocess.run("node --version", shell=True, capture_output=True, text=True).stdout.strip()
    check("Node.js 20+", tuple(int(x.strip('v')) for x in nj.split('.')) >= (20,), nj)
except:
    check("Node.js 20+", False, "not found")

# ffmpeg
try:
    subprocess.run("ffmpeg -version", shell=True, capture_output=True, check=True)
    check("ffmpeg", True)
except:
    check("ffmpeg", False, "install with: apt install ffmpeg")

# aria2
try:
    subprocess.run("aria2c --version", shell=True, capture_output=True, check=True)
    check("aria2c", True)
except:
    check("aria2c", False, "install with: apt install aria2")

# yt-dlp
try:
    yv = subprocess.run("yt-dlp --version", shell=True, capture_output=True, text=True).stdout.strip()
    check("yt-dlp", True, yv)
except:
    check("yt-dlp", False, "will be installed from requirements.txt")

# GPU
try:
    gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                          shell=True, capture_output=True, text=True, check=True).stdout.strip()
    check("GPU", True, gpu)
except:
    check("GPU", False, "CPU mode")

# Disk space
try:
    du = shutil.disk_usage("/content" if os.path.exists("/content") else "/")
    free_gb = du.free / (1024**3)
    check("Disk Space", free_gb > 5, f"{free_gb:.1f} GB free")
except:
    pass

# Credentials
display(Markdown("\n### 🔐 Credentials"))
try:
    from google.colab import userdata
    for name in ['LEECHBOT_API_ID', 'LEECHBOT_API_HASH', 'LEECHBOT_BOT_TOKEN', 'LEECHBOT_USER_ID', 'LEECHBOT_DUMP_ID']:
        try:
            userdata.get(name)
            check(name, True, "set")
        except:
            check(name, False, "not in Secrets")
except ImportError:
    display(Markdown("ℹ️ Colab Secrets not available — fill credentials in Deployer cell"))

display(Markdown("\n---\n> Fix any ❌ above before running the Deployer cell."))